# Lecture 3 (Part 1 of 2) — Flux, Injection & Weighting with **SIREN**

### Neutrino Interactions, Simulation and Event Generation · *N. Kamp*

Lectures 1 & 2 built the physics: how neutrinos **interact** (cross sections, DIS kinematics,
the "Grand Unified Neutrino Spectrum") and how the charged particles they make **propagate and
radiate**. This third lecture is the hands-on **simulation pipeline** that strings those ideas
together:

```
   FLUX            ->   INTERACTION        ->   PROPAGATION + LIGHT + DETECTOR
   what hits us         where/how it scatters   what we actually see
   [        this notebook: SIREN         ]      [ notebook 2: PROPOSAL + Prometheus ]
```

**This is the first of two notebooks.** Here we use **SIREN** (the modern successor to
LeptonInjector) to do the first two stages *fully live*:

1. **Flux** — load the atmospheric flux from SIREN's tables and write down the astrophysical one.
2. **Interaction** — inject high-energy $\nu_\tau$ charged-current events into IceCube.
3. **Weights** — turn that one generated sample into a physical **event rate** for *any* flux.

Everything runs live in Colab from public data — no pre-built cache files. Notebook 2 picks up
the charged lepton (PROPOSAL) and the light it makes in the ice (Prometheus).


## Setup

SIREN is a compiled C++/Python package. The up-to-date version isn't on PyPI yet, so the cell
below installs a **prebuilt manylinux wheel** from this repo's GitHub release (tag `siren-wheels`),
automatically picking the wheel that matches Colab's Python (`cp311`/`cp312`). It installs in
seconds — no source build. We also grab `pyarrow`/`awkward` to read SIREN's event output.

> Once the new SIREN is published to PyPI this whole block collapses to `pip install siren`. The
> wheels were produced from the local source with `tools/build_siren_wheels.sh` (builds inside the
> `manylinux_2_28` container so the result is self-contained and Colab-compatible).

> **⚠️ One-time kernel restart (Colab).** The SIREN wheel pins **numpy 1.x**, which clashes with
> Colab's pre-compiled `awkward`/`pyarrow` (built for numpy 2.x) and would otherwise crash later at
> `ak.from_parquet` (`numpy.dtype size changed, Expected 96 ... got 88`). The cell pins numpy to a
> single ABI and **restarts the kernel once** to apply it. This is expected — when the kernel comes
> back, **just re-run this cell** and it will skip straight through.

Run this cell first. On Colab the disk is wiped each session, so re-run it every time.


In [ ]:
import sys, os, subprocess

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# --- Install SIREN -----------------------------------------------------------
# The up-to-date SIREN isn't on PyPI yet, so we install a prebuilt *manylinux* wheel
# from this repo's GitHub release (tag: siren-wheels), picking the one that matches
# Colab's Python automatically. The wheels were built with tools/build_siren_wheels.sh.
# (Once the new SIREN is published, this whole block becomes simply: pip("siren").)
PYV   = f"cp{sys.version_info.major}{sys.version_info.minor}"        # e.g. cp311 / cp312
WHEEL = f"siren-0.0.3-{PYV}-{PYV}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
REL   = ("https://github.com/nickkamp1/CNP_Neutrino_Interactions/"
         "releases/download/v0.1")


try:
    import siren                       # already available (e.g. on the cluster)? skip install.
    print("siren already installed:", getattr(siren, "__version__", "?"))
except ImportError:
    # Prefer a local wheel if present, else download from the GitHub release.
    local = next((os.path.join(d, WHEEL)
                  for d in ("wheels", "Lecture3_Simulation/wheels", "REPO/Lecture3_Simulation/wheels")
                  if os.path.exists(os.path.join(d, WHEEL))), None)
    src = local or f"{REL}/{WHEEL}"
    print("installing SIREN from:", src)
    try:
        pip(src)
    except subprocess.CalledProcessError:
        raise RuntimeError(
            f"Failed to install {WHEEL}. Make sure a wheel for {PYV} exists in the "
            f"'siren-wheels' release (check !python --version vs the wheel's cpXY tag), "
            f"or build one with tools/build_siren_wheels.sh.")
    import siren

# --- numpy ABI fix (Colab) ---------------------------------------------------
# Installing the SIREN wheel can pull a different numpy (it pins numpy 1.x), which then
# disagrees with Colab's pre-compiled awkward/pyarrow (built for numpy 2.x). The mismatch
# crashes later at ak.from_parquet with:
#     "numpy.dtype size changed, Expected 96 ... got 88"   (96 = numpy 2.x ABI, 88 = 1.x)
# Fix: pin numpy to the version SIREN's wheel was built against, reinstall awkward/pyarrow
# against it so every compiled package shares ONE ABI, then restart the kernel once. The
# cached re-run (numpy already == NUMPY_PIN) skips straight through. If your wheel was built
# against a different numpy, change NUMPY_PIN to match it.
NUMPY_PIN = "1.26.4"
import numpy
if numpy.__version__ != NUMPY_PIN:
    print(f"pinning numpy {numpy.__version__} -> {NUMPY_PIN} for a consistent ABI ...")
    pip("--force-reinstall", f"numpy=={NUMPY_PIN}")
    pip("--force-reinstall", "pyarrow", "awkward")   # rebuilt against the pinned numpy
    print("Restarting the kernel once so the new numpy ABI takes effect.\n"
          ">>> When it comes back up, just RE-RUN this cell (it will skip straight through). <<<")
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(True)   # Jupyter / Colab
    except Exception:
        os.kill(os.getpid(), 9)                                   # hard fallback (Colab auto-restarts)
    raise SystemExit("Kernel restarting -- re-run this cell.")    # stop before the bad-ABI imports

# Reached only once numpy/awkward/pyarrow share one ABI (after the restart). Ensure the
# event readers are present on the fresh kernel.
pip("pyarrow", "awkward")

import numpy as np
import matplotlib.pyplot as plt
import awkward as ak

print("python:", PYV, "| siren version:", getattr(siren, "__version__", "?"))
print("numpy:", np.__version__, "| siren importable:", True)


### The bundled download scripts

SIREN's detectors, cross-section splines, and flux tables are **data files** hosted on Zenodo
(record `20129082`) and a companion GitHub data repo. They are *not* shipped inside the wheel —
SIREN downloads them on demand the first time a loader needs them. SIREN exposes a small CLI,
**`siren-download`**, to inspect and pre-fetch that data. Let's use it explicitly.

First, list everything SIREN knows how to load:


In [ ]:
# What detectors / fluxes / cross-section models are available?
subprocess.run(["siren-download", "--list"], check=True)


Now **pre-fetch** exactly what this notebook needs — the CSMS deep-inelastic cross-section
splines and the atmospheric flux tables. (If you skip this, the loaders below would download the
same files automatically the first time they run; doing it up front just makes the data step
explicit.)

> **Why this stays small.** The cross-section archive on Zenodo bundles *many* process models.
> SIREN's `ensure_zenodo_archive(...)` helper does **prefix extraction**: it pulls the one zip
> from Zenodo but only unpacks the entries under the prefix you asked for (e.g. just
> `CSMSDISSplines`), so you don't pay disk/time for the dipole-portal, HNL, MARLEY, … tables you
> aren't using.


In [ ]:
# Pre-fetch the two resources we use. Each prints a download/extract progress line.
subprocess.run(["siren-download", "--fetch", "CSMSDISSplines"], check=True)
subprocess.run(["siren-download", "--fetch", "Atmospheric"], check=True)
print("\nData ready.")


## 1. The flux — what's hitting the detector?

A neutrino telescope's "menu" is dominated by two diffuse fluxes (see **Lecture 1: the Grand
Unified Neutrino Spectrum** — the eV-to-EeV census of every neutrino source):

1. **Atmospheric** $\nu$ from cosmic-ray air showers — steeply falling ($\sim E^{-3.7}$ at high
   energy), the dominant background below $\sim$100 TeV.
2. **Astrophysical** $\nu$ — a near-isotropic diffuse flux from distant accelerators, well
   described by a single **power law** $\propto E^{-\gamma}$.

The flux is the first factor in the master formula every counting experiment uses (**Lecture 1**):

$$ \text{Rate} \;=\; \underbrace{\Phi(E,\theta)}_{\text{flux}} \;\times\; \underbrace{\sigma(E)}_{\text{cross section}} \;\times\; \underbrace{N_{\text{targets}}}_{\text{detector}} \;\times\; \underbrace{\varepsilon}_{\text{efficiency}} $$

SIREN ships the **atmospheric** flux as tabulated tables (several hadronic models, both at the
**surface** and **at the detector** after Earth propagation). The astrophysical flux is so simple
we just write it analytically.


In [ ]:
# --- Atmospheric flux from SIREN's tables ------------------------------------
# load_flux returns a PATH to a text table with columns: [E_GeV, cos(zenith), flux].
# Tag format:  {model}_{osc|unosc}_{species1}_{species2}_...
#   model in {bartol, daemonflux, H3a_SIBYLL21, H3a_SIBYLL23C, honda2006}
#   unosc = at the surface,  osc = at the detector (nuSQuIDS oscillation + Earth absorption folded in)
MODEL = "daemonflux"

def load_atmo_table(tag):
    path = siren.utilities.load_flux("Atmospheric", tag=tag)
    d = np.loadtxt(path)                      # columns: E, cos_zenith, flux
    E   = np.unique(d[:, 0])
    cz  = np.unique(d[:, 1])
    # The table is written cos_zenith-major (cos_zenith fixed, energy sweeps),
    # so reshape (n_cz, n_E) then transpose to get [energy, cos_zenith].
    flux = d[:, 2].reshape(len(cz), len(E)).T
    return E, cz, flux

# numu + numubar, surface vs at-detector
E, cz, flux_surf = load_atmo_table(f"{MODEL}_unosc_numu_numubar")
_, _, flux_det   = load_atmo_table(f"{MODEL}_osc_numu_numubar")
print(f"atmospheric table: {len(E)} energies x {len(cz)} zenith angles")
print(f"E range {E.min():.2g}-{E.max():.2g} GeV,  cos(zenith) {cz.min()}..{cz.max()}")


### 1a. The two fluxes a telescope sees

Average the atmospheric table over the sky (it's nearly isotropic at the high energies we care
about) and overlay the analytic astrophysical power law. Plotting $E^2\,d\Phi/dE$ flattens the
steep spectra so the **crossover** — where astrophysical overtakes atmospheric — is easy to read.

> **A note on normalization.** SIREN's tabulated atmospheric flux and our hand-written astrophysical
> power law are carried in *different unit conventions*, so a raw overlay isn't apples-to-apples.
> Here we rescale the atmospheric curve to a standard physical anchor (conventional $\nu_\mu$,
> $E^2\Phi \approx 2\times10^{-5}\,$GeV cm$^{-2}$s$^{-1}$sr$^{-1}$ at 1 TeV) so the **shapes and their
> crossover** — the physics point — are directly comparable. The absolute astrophysical normalization
> is the IceCube diffuse fit.


In [ ]:
def astro_powerlaw(E_gev, phi0=1.8e-18, gamma=2.5, E0=1e5):
    '''IceCube-style single power-law astrophysical flux (per flavor, nu+nubar),
    units GeV^-1 cm^-2 s^-1 sr^-1.  Defaults ~ IceCube diffuse fit.'''
    return phi0 * (np.asarray(E_gev, float) / E0) ** (-gamma)

# Sky-averaged atmospheric numu (at detector):
phi_atmo_raw = flux_det.mean(axis=1)         # average over cos(zenith), table units

# Rescale the atmospheric table to a physical anchor at 1 TeV (shape preserved):
i1tev = np.argmin(np.abs(E - 1e3))
anchor = 2.2e-5                                # E^2 * dPhi/dE at 1 TeV [GeV cm^-2 s^-1 sr^-1]
scale = anchor / (E[i1tev]**2 * phi_atmo_raw[i1tev])
phi_atmo = phi_atmo_raw * scale

phi_astro = astro_powerlaw(E, gamma=2.5)

plt.figure(figsize=(7, 5))
plt.loglog(E, E**2 * phi_atmo,  label=r"atmospheric $\nu_\mu+\bar\nu_\mu$ (sky avg, at detector)")
plt.loglog(E, E**2 * phi_astro, label=r"astrophysical ($\gamma=2.5$)")
plt.xlabel("Neutrino energy [GeV]")
plt.ylabel(r"$E^2\,d\Phi/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]")
plt.xlim(1e2, 1e7); plt.ylim(1e-9, 1e-3)
plt.legend(); plt.grid(alpha=0.3)
plt.title("The two fluxes a neutrino telescope sees")
plt.show()

# Where does astro overtake atmospheric?  Find the sign change of (astro - atmo) above 1 TeV.
hi = E > 1e3
diff = phi_astro[hi] - phi_atmo[hi]
sign_change = np.where(np.diff(np.sign(diff)) > 0)[0]
cross = E[hi][sign_change[0]] if len(sign_change) else np.nan
print(f"astrophysical overtakes atmospheric near {cross/1e3:.0f} TeV "
      "(the energy above which a telescope's signal is mostly astrophysical)")


> **★ Discussion.** Change `gamma` from `2.5` to `2.0` (a *harder* astrophysical spectrum) and
> re-run. Which way does the crossover move, and why does a harder spectrum make high-energy events
> relatively more common? (This is exactly the lever that sets the energy threshold of a diffuse
> astrophysical search.)


### 1b. What the Earth does: surface vs at-detector

SIREN ships **both** the surface flux (`unosc`) and the at-detector flux (`osc`, already propagated
through the Earth with nuSQuIDS). Taking the **ratio** of the two reveals two physical effects at a
glance — without us running any oscillation/absorption code ourselves. We do this for $\nu_\tau$,
where the effect is most dramatic:

- **Oscillations** (tens of GeV): atmospheric $\nu_\mu \to \nu_\tau$ **appearance** — there is almost
  no $\nu_\tau$ produced in air showers, so *any* low-energy $\nu_\tau$ at the detector got there by
  oscillating. Ratio $\gg 1$.
- **Earth absorption** (above $\sim$100 TeV): up-going neutrinos traverse a lot of rock and start to
  be absorbed by CC interactions. Ratio $< 1$ for up-going.


In [ ]:
# nu_tau + nu_taubar, surface vs at-detector, across the (E, cos-zenith) plane
Et, czt, tau_surf = load_atmo_table(f"{MODEL}_unosc_nutau_nutaubar")
_,  _,   tau_det  = load_atmo_table(f"{MODEL}_osc_nutau_nutaubar")
ratio = np.divide(tau_det, tau_surf, out=np.full_like(tau_det, np.nan), where=tau_surf > 0)

plt.figure(figsize=(7.5, 5))
pc = plt.pcolormesh(Et, czt, ratio.T, shading="auto", vmin=0, vmax=3, cmap="RdBu_r")
plt.xscale("log"); plt.xlim(1, 1e7)
plt.xlabel("Neutrino energy [GeV]")
plt.ylabel(r"$\cos\theta_{\rm zenith}$   ($-1$ = up-going, through the Earth)")
plt.colorbar(pc, label=r"$\nu_\tau$ flux:  detector / surface")
plt.title(r"Earth effects on atmospheric $\nu_\tau$: appearance (low E) + absorption (high E)")
plt.show()
print("ratio >> 1 at low E  -> nu_tau APPEARANCE via oscillation")
print("ratio  < 1 up-going at high E -> Earth ABSORPTION")


## 2. The interaction — inject neutrinos and let them scatter

Now the neutrinos reach the ice and interact. Event generators handle the cross section + the
final-state kinematics. The landscape (see **Lecture 1**):

| Generator | Regime / role |
|---|---|
| **GENIE** (+ NuWro, GiBUU) | MeV–few GeV: quasi-elastic, resonant, nuclear effects |
| **NuGen** | IceCube's high-energy DIS workhorse |
| **LeptonInjector** | volume injection + weights at telescope scale |
| **SIREN** | the modern successor to LeptonInjector: flexible processes + weighting |

We use **SIREN** to inject high-energy $\nu_\tau$ **charged-current deep-inelastic** events into
IceCube, using the **CSMS** DIS cross-section splines. The key telescope trick is **weighted
(forced) injection**: instead of throwing $\sim 10^{12}$ neutrinos and keeping the rare one that
interacts, SIREN *forces* an interaction near the detector and records a **generation weight** that
encodes how improbable that was. We turn those weights into a real rate in §3.

This pattern follows SIREN's own `resources/examples/example1/DIS_IceCube.py`.


In [ ]:
from siren._util import GenerateEvents, SaveEvents

events_to_inject = 3000        # a few thousand -> runs in well under a minute on Colab CPU

# 1) Detector + primary particle
detector_model = siren.utilities.load_detector("IceCube")
primary_type = siren.dataclasses.Particle.ParticleType.NuTau          # <- tau neutrino!
Nucleon      = siren.dataclasses.Particle.ParticleType.Nucleon

# 2) CC DIS cross sections (CSMS splines that ship with SIREN)
primary_processes, _ = siren.utilities.load_processes(
    "CSMSDISSplines",
    primary_types=[primary_type],
    target_types=[Nucleon],
    isoscalar=True,
    process_types=["CC"],
)
primary_cross_sections = primary_processes[primary_type]

# 3) Injector: forces an interaction in/near the detector and tracks a generation weight
injector = siren.injection.Injector()
injector.number_of_events  = events_to_inject
injector.detector_model    = detector_model
injector.primary_type      = primary_type
injector.primary_interactions = primary_cross_sections
injector.primary_injection_distributions = [
    siren.distributions.PrimaryMass(0),
    siren.distributions.PowerLaw(2, 1e4, 1e7),          # inject E ~ E^-2 over 10 TeV - 10 PeV
    siren.distributions.IsotropicDirection(),
    siren.distributions.ColumnDepthPositionDistribution(
        600, 600.0, siren.distributions.LeptonDepthFunction()),
]

events, gen_times = GenerateEvents(injector)
print(f"\ngenerated {len(events)} nu_tau CC-DIS events")


### 2a. Build a Weighter and write the events to disk

The **Weighter** turns each generated event into a physical rate for a chosen flux. We give it the
same injection setup plus the *physical* distributions we want to weight to (we start with the
injection spectrum; in §3 we'll swap in the real fluxes). `SaveEvents` then writes a parquet file
of per-event kinematics + the event weight, which we read back with `awkward`.


In [ ]:
weighter = siren.injection.Weighter()
weighter.injectors = [injector]
weighter.detector_model = detector_model
weighter.primary_type = primary_type
weighter.primary_interactions = primary_cross_sections
weighter.primary_physical_distributions = [
    siren.distributions.PowerLaw(2, 1e4, 1e7),
    siren.distributions.IsotropicDirection(),
]

# Save -> parquet (we skip the hdf5 and custom siren-event formats to keep it light)
SaveEvents(events, weighter, gen_times,
           save_hdf5=False, save_siren_events=False,
           output_filename="nutau_icecube")

data = ak.from_parquet("nutau_icecube.parquet")
print("\nfields:", data.fields)


### 2b. Injected energy and the inelasticity $y$

From the saved event tree we pull, per event:
- the **incoming neutrino energy** $E_\nu$ (the primary momentum's 0th component),
- the **hadronic energy** $E_{\rm had}$ (the `Hadrons` secondary), which defines the **Bjorken
  inelasticity** $y \equiv E_{\rm had}/E_\nu$ — the fraction of energy transferred to the hadronic
  cascade (**Lecture 1: DIS kinematics**). The outgoing lepton carries the rest, $(1-y)E_\nu$.


In [ ]:
HADRONS = -2000001006     # SIREN particle code for the hadronic system
TAU     = 15              # TauMinus

Enu, y = [], []
for ie in range(len(data)):
    types = np.array(data["secondary_types"][ie][0])
    moms  = np.array(data["secondary_momenta"][ie][0])     # rows: [E, px, py, pz]
    E_nu  = np.array(data["primary_momentum"][ie][0])[0]
    E_had = moms[types == HADRONS][0][0]
    Enu.append(E_nu)
    y.append(E_had / E_nu)
Enu = np.array(Enu); y = np.array(y)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist(np.log10(Enu), bins=40, color="C0")
ax[0].set_xlabel(r"$\log_{10}(E_\nu / {\rm GeV})$")
ax[0].set_ylabel("generated events")
ax[0].set_title("Injected energy spectrum (pre-weight)")

ax[1].hist(y, bins=40, range=(0, 1), color="C1")
ax[1].set_xlabel(r"inelasticity  $y = E_{\rm had}/E_\nu$")
ax[1].set_ylabel("generated events")
ax[1].set_title(r"Inelasticity: $E_{\rm had}=y\,E_\nu$ -> hadronic cascade")
plt.tight_layout(); plt.show()

print(f"median inelasticity <y> = {np.median(y):.2f}")


> **★ Discussion.** At these energies the median inelasticity is $\langle y\rangle \approx 0.25$,
> i.e. the **outgoing lepton** keeps most of the energy. Two follow-ups tied to the lectures:
>
> 1. **(Lecture 1, $F_3$)** Neutrinos and antineutrinos have *different* inelasticity distributions
>    because the parity-violating structure function $F_3$ enters with opposite sign — $\bar\nu$
>    events are pushed to lower $y$. Re-run §2 with `primary_type = ...ParticleType.NuTauBar` and
>    overlay the two $y$ histograms. Which one is more sharply peaked at low $y$?
> 2. **(Lecture 2, the tau)** The lepton energy $(1-y)E_\nu$ is what powers the $\nu_\tau$ "double
>    bang": the first cascade is the hadronic $yE_\nu$; the tau then travels $\sim 50\,{\rm m/PeV}$
>    and decays into a *second* cascade. Histogram $(1-y)\,E_\nu$ here — that's the input to the
>    PROPOSAL study in **notebook 2**.


## 3. Event weights — from generated events to a physical rate

We injected events on an $E^{-2}$ spectrum *just to populate the detector evenly* across energy —
that shape is **not** physical. Each event carries a **generation weight**, and to predict what a
detector actually sees we reweight to a physical flux $\Phi$:

$$ \frac{dN}{dt} \;=\; \sum_{\text{events}} w_{\rm gen}\;\times\;\Phi(E,\theta)\;\times\;\sigma(E)\;\times\;(\text{geometry}) $$

**Why weighted generation is powerful:** one expensive event sample can be reweighted to *any* flux
model — change $\gamma$, swap astrophysical for atmospheric — by changing only $\Phi$ in the sum.
No re-simulating the costly photon propagation (notebook 2). This is the "benefits of weighted event
generation" point from the Lecture 3 outline.

In SIREN, $\Phi$ lives in the Weighter's `primary_physical_distributions`. We can rebuild the
Weighter with different physical fluxes and reweight the **same events**.


In [ ]:
# The Weighter already gives w(event) = w_gen * Phi_phys * sigma * geometry for whatever
# physical distribution it was built with. Our weighter used PowerLaw(2,...) as Phi_phys, so:
#
#     w(event) / Phi_phys(E)  =  w_gen * sigma * geometry        (the flux-independent part)
#
# We can therefore reweight to ANY flux Phi_new(E) by multiplying that flux-independent part by
# Phi_new(E).  This is exactly the "one sample, any flux" trick.

w_phys = np.array(data["event_weight"])             # weight under the injection-spectrum flux
def powerlaw(E, gamma, E0=1e5, norm=1.0):
    return norm * (E / E0) ** (-gamma)

# strip off the PowerLaw(gamma=2) physical flux that the weighter used:
phi_inj = powerlaw(Enu, gamma=2.0)
w_kernel = w_phys / phi_inj                          # flux-independent: w_gen * sigma * geometry

print(f"total weight under E^-2 physical flux: {w_phys.sum():.3g}")
print("w_kernel (flux-independent) ready for reweighting.")


### 3a. Reweight the same sample to different fluxes

Now multiply the flux-independent kernel by different physical fluxes and histogram the resulting
**rate vs energy**. We compare two astrophysical spectral indices and the atmospheric $\nu_\tau$
table from §1.


In [ ]:
from scipy.interpolate import RegularGridInterpolator

# event directions: cos(zenith) = pz / |p|
cz_ev = np.array([np.array(data["primary_momentum"][ie][0])[3] /
                  np.linalg.norm(np.array(data["primary_momentum"][ie][0])[1:])
                  for ie in range(len(data))])

# atmospheric nu_tau at-detector flux, interpolated at each event's (cos-zenith, log10 E).
# Rescale to the same physical anchor used in 1a (shape preserved; absolute table units differ).
tau_skyavg = tau_det.mean(axis=1)
i1 = np.argmin(np.abs(Et - 1e3))
tau_scale = anchor / (Et[i1]**2 * tau_skyavg[i1])
interp = RegularGridInterpolator(
    (czt, np.log10(Et)), (tau_det * tau_scale).T, bounds_error=False, fill_value=0.0)
czc = np.clip(cz_ev, czt.min(), czt.max())
phi_atmo_ev = interp(np.column_stack([czc, np.log10(np.clip(Enu, Et.min(), Et.max()))]))

bins = np.linspace(4, 7, 25)                          # log10(E/GeV), 10 TeV - 10 PeV
def rate_curve(flux_ev, label, **kw):
    h, edges = np.histogram(np.log10(Enu), bins=bins, weights=w_kernel * flux_ev)
    c = 0.5 * (edges[1:] + edges[:-1])
    plt.step(c, h, where="mid", label=label, **kw)

plt.figure(figsize=(7.5, 5))
for gamma in (2.0, 2.5):
    rate_curve(astro_powerlaw(Enu, gamma=gamma), f"astro $\\gamma={gamma}$")
rate_curve(phi_atmo_ev, r"atmospheric $\nu_\tau$ (at detector)", ls="--", color="k")

plt.yscale("log")
plt.xlabel(r"$\log_{10}(E_\nu / {\rm GeV})$")
plt.ylabel(r"expected rate  $dN/d\log_{10}E$  [a.u.]")
plt.legend(); plt.grid(alpha=0.3)
plt.title(r"One sample, three fluxes: astrophysical vs atmospheric $\nu_\tau$")
plt.show()


### 3b. A single number: the event rate

Summing a weighted sample gives an **expected number of events** (here in arbitrary units, since we
are only carrying the flux/cross-section/geometry factors and not a livetime $\times$ detection
efficiency). The point is the *machinery*: change the flux, get a new rate, instantly.


In [ ]:
rate_astro_25 = np.sum(w_kernel * astro_powerlaw(Enu, gamma=2.5))
rate_astro_20 = np.sum(w_kernel * astro_powerlaw(Enu, gamma=2.0))
rate_atmo     = np.sum(w_kernel * phi_atmo_ev)

print("Expected rates (same generated sample, different physical fluxes; a.u.):")
print(f"  astrophysical  gamma=2.5 : {rate_astro_25:.3e}")
print(f"  astrophysical  gamma=2.0 : {rate_astro_20:.3e}")
print(f"  atmospheric    nu_tau    : {rate_atmo:.3e}")
print("\nKey point: ONE generated sample, computed three rates by changing only the flux factor")
print("in the weighted sum -- no re-simulation. That is the payoff of weighted event generation.")


> **★ Discussion.** The dashed atmospheric $\nu_\tau$ curve falls steeply and crosses the
> astrophysical curves; above that energy a detected $\nu_\tau$ is *more likely astrophysical than
> atmospheric*. (i) Read off the crossover energy. (ii) Make the astrophysical spectrum harder
> ($\gamma=2.0$): does the crossover — i.e. the energy threshold for a clean tau-appearance search —
> move up or down? Normalizations here are arbitrary, so it's the **shape and crossover** that carry
> the physics.


## Wrap-up & exercises

You ran the first half of the telescope pipeline **fully live**: SIREN downloaded its own
cross-section and flux data, injected $\nu_\tau$ CC-DIS events into IceCube, and reweighted that one
sample to multiple physical fluxes.

**★ Exercises**

1. **Antineutrinos & $F_3$ (Lecture 1).** Re-run §2 with `NuTauBar` and overlay the inelasticity
   distributions of $\nu_\tau$ vs $\bar\nu_\tau$. Confirm the $\bar\nu$ distribution is pushed to
   lower $y$.
2. **Hand the tau to notebook 2.** Histogram the outgoing tau energy $(1-y)\,E_\nu$ from §2b — this
   is the initial energy distribution that feeds the PROPOSAL decay-length study in notebook 2.
3. **Swap the physical flux.** Rebuild the SIREN `Weighter` with a tabulated atmospheric flux
   (`siren.distributions.Tabulated2DFluxDistribution(E_min, E_max, flux_file, True)` using a path
   from `siren.utilities.load_flux`) instead of doing the kernel trick by hand, and check you get
   the same atmospheric rate.
4. **Astro vs atmo crossover.** Scan $\gamma$ from 2.0 to 3.0 and plot the astrophysical/atmospheric
   crossover energy vs $\gamma$.

**References.** SIREN / LeptonInjector (arXiv:2012.10449); SIREN data on Zenodo (record 20129082);
CSMS cross sections (Cooper-Sarkar, Mertsch, Sarkar, JHEP 2011); Formaggio & Zeller, Rev. Mod. Phys.
84 (2012) for the cross-section + DIS kinematics review.

**Next:** notebook 2 — `2_propagation_and_light.ipynb` — propagates the tau with **PROPOSAL** and
turns the cascade(s) into Cherenkov light and a detector event display with **Prometheus**.
